# ComfyUI de Leo — Z-Image Turbo (Colab)

Notebook propio para correr ComfyUI con Z-Image Turbo + ControlNet + upscale, manejado desde **comfyweb**.

**Cómo usarlo:** arriba, *Entorno de ejecución → Ejecutar todo*. La 1ª vez baja los modelos a tu Google Drive (~14 GB, tarda); después ya quedan y arranca en ~2-3 min.

Al final imprime una **URL pública** — pegála en comfyweb (botón *Colab (Cloudflare)*).

> Para una URL **fija** (que no cambie cada vez) ver `README.md` → sección *Túnel fijo*.

## 1) Montar Google Drive (los modelos viven acá, no se rebajan)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

# Carpeta tuya en Drive (se crea sola la 1ª vez)
DRIVE_BASE = '/content/drive/MyDrive/ComfyUI-Leo'
MODELS = os.path.join(DRIVE_BASE, 'models')
for sub in ['diffusion_models','text_encoders','vae','model_patches','upscale_models','loras']:
    os.makedirs(os.path.join(MODELS, sub), exist_ok=True)
print('Drive montado. Modelos en:', MODELS)

## 2) Clonar ComfyUI + custom nodes (Z-Image + ControlNet + upscale)

In [ ]:
import os
%cd /content
if not os.path.isdir('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt

# Custom nodes que usa el pipeline DIPLO (ControlNet preprocesadores, tiles, etc.)
CN = '/content/ComfyUI/custom_nodes'
repos = [
    'https://github.com/ltdrdata/ComfyUI-Manager',
    'https://github.com/Fannovel16/comfyui_controlnet_aux',   # AIO_Preprocessor
    'https://github.com/TTPlanetPig/Comfyui_TTP_Toolset',     # tiles
    'https://github.com/yolain/ComfyUI-Easy-Use',
    'https://github.com/rgthree/rgthree-comfy',
    'https://github.com/LAOGOU-666/Comfyui-Memory_Cleanup',   # RAMCleanup
]
for u in repos:
    name = u.rstrip('/').split('/')[-1]
    p = os.path.join(CN, name)
    if not os.path.isdir(p):
        !git clone --depth 1 {u} {p}
    req = os.path.join(p, 'requirements.txt')
    if os.path.isfile(req):
        !pip install -q -r {req}
print('\nComfyUI + custom nodes listos')

# --- Mini endpoint para agregar LoRAs por URL desde comfyweb (POST /leo/add_lora) ---
import os as _os
_os.makedirs('/content/ComfyUI/custom_nodes/leo_lora_fetch', exist_ok=True)
with open('/content/ComfyUI/custom_nodes/leo_lora_fetch/__init__.py','w') as _f:
    _f.write(r"""
import os, urllib.request
from aiohttp import web
import server, folder_paths

@server.PromptServer.instance.routes.post("/leo/add_lora")
async def leo_add_lora(request):
    try:
        data = await request.json()
    except Exception:
        data = {}
    url = (data.get("url") or "").strip()
    filename = (data.get("filename") or "").strip()
    if not url:
        return web.json_response({"success": False, "error": "falta url"}, status=400)
    dirs = folder_paths.get_folder_paths("loras")
    target_dir = dirs[0] if dirs else "/content/ComfyUI/models/loras"
    os.makedirs(target_dir, exist_ok=True)
    if not filename:
        filename = url.split("?")[0].rstrip("/").split("/")[-1] or "lora.safetensors"
    if not filename.lower().endswith((".safetensors", ".pt", ".ckpt", ".bin")):
        filename += ".safetensors"
    dest = os.path.join(target_dir, filename)
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=600) as r, open(dest, "wb") as out:
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                out.write(chunk)
    except Exception as e:
        return web.json_response({"success": False, "error": str(e)}, status=500)
    try:
        if hasattr(folder_paths, "filename_list_cache"):
            folder_paths.filename_list_cache.clear()
    except Exception:
        pass
    return web.json_response({"success": True, "name": filename, "path": dest})

NODE_CLASS_MAPPINGS = {}
NODE_DISPLAY_NAME_MAPPINGS = {}
""")
print('leo_lora_fetch instalado (endpoint /leo/add_lora)')


## 3) Descargar modelos a Drive (solo la 1ª vez)

Si ya están en tu Drive, los saltea. Fuentes oficiales (Apache-2.0).

In [ ]:
import os

def dl(url, dest, min_mb=1):
    if os.path.isfile(dest) and os.path.getsize(dest) > min_mb*1_000_000:
        print('✓ ya está:', os.path.basename(dest)); return
    print('↓ bajando:', os.path.basename(dest))
    !wget -q --show-progress -O "{dest}" "{url}"

MODELS = '/content/drive/MyDrive/ComfyUI-Leo/models'

# Difusión (Z-Image Turbo fp8 e4m3fn, ideal para T4)
dl('https://huggingface.co/T5B/Z-Image-Turbo-FP8/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors',
   f'{MODELS}/diffusion_models/z-image-turbo-fp8-e4m3fn.safetensors', 500)
# Text encoder (Qwen3-4B fp8) — CLIPLoader tipo lumina2
dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b_fp8_mixed.safetensors',
   f'{MODELS}/text_encoders/qwen_3_4b_fp8_mixed.safetensors', 500)
# VAE
dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors',
   f'{MODELS}/vae/ae.safetensors', 50)
# ControlNet Union (Canny/Depth/Pose/etc.)
dl('https://huggingface.co/alibaba-pai/Z-Image-Turbo-Fun-Controlnet-Union/resolve/main/Z-Image-Turbo-Fun-Controlnet-Union.safetensors',
   f'{MODELS}/model_patches/Z-Image-Turbo-Fun-Controlnet-Union.safetensors', 500)
# Upscaler 2x
dl('https://huggingface.co/Kim2091/2x-AnimeSharpV4/resolve/main/2x-AnimeSharpV4_RCAN.safetensors',
   f'{MODELS}/upscale_models/2x-AnimeSharpV4_RCAN.safetensors', 5)

print('\nModelos OK en Drive')

## 4) Apuntar ComfyUI a los modelos de Drive + lanzar + túnel público

Al final imprime la **URL pública**. Pegála en comfyweb.

In [ ]:
# ComfyUI lee los modelos desde Drive (no copia, los usa en su lugar)
yaml = '''leo_drive:
  base_path: /content/drive/MyDrive/ComfyUI-Leo/models
  diffusion_models: diffusion_models
  text_encoders: text_encoders
  clip: text_encoders
  vae: vae
  model_patches: model_patches
  upscale_models: upscale_models
  loras: loras
'''
open('/content/ComfyUI/extra_model_paths.yaml','w').write(yaml)
print('extra_model_paths.yaml escrito')

# ============================================================
#  TUNEL FIJO -> https://comfy.leoblumfeld.com  (Cloudflare named tunnel)
#  Requisito (1 sola vez): en Colab, panel izq -> 🔑 Secretos ->
#    crear secreto  CF_TUNNEL_CRED  con el JSON de credencial del tunnel
#    (cuidado: es secreto, NO va al repo). Activá el acceso del notebook.
#  Si NO está el secreto, cae al tunel random (pycloudflared) como respaldo.
# ============================================================
import threading, time, socket, subprocess, os
%cd /content/ComfyUI

CRED = None
try:
    from google.colab import userdata
    CRED = userdata.get('CF_TUNNEL_CRED')
except Exception as e:
    print('Sin secreto CF_TUNNEL_CRED ->', e)

def wait_8188():
    while socket.socket().connect_ex(('127.0.0.1', 8188)) != 0:
        time.sleep(1)

if CRED:
    open('/content/comfy-leo.json','w').write(CRED)
    if not os.path.isfile('/usr/local/bin/cloudflared'):
        os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared')
    URL = 'https://comfy.leoblumfeld.com'
    def tunnel():
        wait_8188()
        print('\n\n' + '='*48)
        print('  URL FIJA (pegala en comfyweb):')
        print('  ' + URL)
        print('='*48 + '\n')
        subprocess.run(['cloudflared','tunnel','--no-autoupdate',
                        '--url','http://127.0.0.1:8188',
                        'run','--credentials-file','/content/comfy-leo.json',
                        'comfy-leo'])
    threading.Thread(target=tunnel, daemon=True).start()
else:
    # Respaldo: tunel random cada arranque
    os.system('pip install -q pycloudflared')
    def tunnel():
        wait_8188()
        from pycloudflared import try_cloudflare
        url = try_cloudflare(port=8188).tunnel
        print('\n\n' + '='*48)
        print('  URL PUBLICA (random, pegala en comfyweb):')
        print('  ' + url)
        print('='*48 + '\n')
    threading.Thread(target=tunnel, daemon=True).start()

!python main.py --dont-print-server

---
### Credencial del túnel fijo (referencia)

La URL fija `https://comfy.leoblumfeld.com` ya está configurada en Cloudflare
(túnel **comfy-leo**, DNS ruteado). Sólo falta el secreto en Colab:

1. Panel izquierdo → **🔑 Secretos** → **+ Agregar secreto nuevo**
2. Nombre: `CF_TUNNEL_CRED`
3. Valor: el JSON de credencial del túnel (lo tenés en
   `~/.cloudflared/01c8bdfb-...json` en la compu de Leo). **Es secreto, no lo subas al repo.**
4. Activá el toggle de *Acceso del notebook*.

Con eso, la celda 4 levanta el túnel fijo solo. Sin el secreto, usa un túnel random de respaldo.